# VoltIQ – 04: Feature Preparation

**Objective**: Derive engineered features required by downstream ML models, API endpoints, and dashboard visualisations.

> No model training occurs here. This notebook only creates derived columns and documents their definitions.

## 1. Setup & Imports

In [ ]:
import sys, os, logging
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(level=logging.INFO,
    format='[%(levelname)s] %(name)s — %(message)s')

from app.utils.data_loader import DataLoader
from app.utils.cleaning import DataCleaner
from app.utils.feature_engineering import FeatureEngineer

loader  = DataLoader()
cleaner = DataCleaner()
fe      = FeatureEngineer()

raw_dfs   = loader.load_all()
clean_dfs = cleaner.clean_all(raw_dfs)
print('Clean datasets ready:', list(clean_dfs.keys()))

## 2. Engineer All Features

In [ ]:
engineered_dfs = fe.engineer_all(clean_dfs)
print(f'\nFeature engineering complete for: {list(engineered_dfs.keys())}')

## 3. Fleet Readiness — New Features

In [ ]:
fleet_fe = engineered_dfs['fleet_readiness']
new_fleet_cols = ['Vehicle_Age_Category', 'Maintenance_Cost_Band',
                  'Fuel_Efficiency_Category', 'Load_Utilization_Category',
                  'Readiness_Label']
print('New fleet columns:')
for col in new_fleet_cols:
    if col in fleet_fe.columns:
        print(f'\n  {col}:')
        print(fleet_fe[col].value_counts().to_string())

## 4. Battery — New Features

In [ ]:
battery_fe = engineered_dfs['battery']
new_battery_cols = ['Battery_Age_Indicator', 'SOH_Category',
                    'Degradation_Severity', 'Capacity_Loss_Pct']
print('New battery columns:')
for col in new_battery_cols:
    if col in battery_fe.columns:
        print(f'\n  {col}:')
        print(battery_fe[col].value_counts().to_string())

## 5. Charging Sessions — New Features

In [ ]:
charging_fe = engineered_dfs['charging_sessions']
new_charge_cols = ['Session_Duration_Computed_hrs',
                   'Charging_Efficiency_Computed_Pct',
                   'Fast_Charging_Flag', 'SOC_Delta_Computed',
                   'Cost_Per_kWh_Computed']
print('New charging session columns:')
for col in new_charge_cols:
    if col in charging_fe.columns:
        print(f'  {col}: mean={charging_fe[col].mean():.3f}')

## 6. Carbon Intelligence — New Features

In [ ]:
carbon_fe = engineered_dfs['carbon']
new_carbon_cols = ['Carbon_Saving_Potential_kg', 'Net_Zero_Progress_Category',
                   'High_Emitter_Flag', 'EV_Priority_Score']
print('New carbon columns:')
for col in new_carbon_cols:
    if col in carbon_fe.columns:
        print(f'\n  {col}:')
        if carbon_fe[col].dtype == bool:
            print(f'    True: {carbon_fe[col].sum():,}')
        elif carbon_fe[col].dtype == 'object':
            print(carbon_fe[col].value_counts().to_string())
        else:
            print(f'    mean={carbon_fe[col].mean():.2f}, max={carbon_fe[col].max():.2f}')

## 7. Weather — New Features

In [ ]:
weather_fe = engineered_dfs['weather']
new_weather_cols = ['Temperature_Category', 'Climate_Severity_Index',
                    'Extreme_Weather_Flag']
print('New weather columns:')
for col in new_weather_cols:
    if col in weather_fe.columns:
        print(f'\n  {col}:')
        if weather_fe[col].dtype == bool:
            print(f'    Extreme events: {weather_fe[col].sum():,}')
        else:
            print(weather_fe[col].value_counts().head(5).to_string())

## 8. Column Count Comparison

In [ ]:
import pandas as pd

comparison = []
for key in clean_dfs:
    if key in engineered_dfs:
        comparison.append({
            'Dataset':     key,
            'Before Cols': len(clean_dfs[key].columns),
            'After Cols':  len(engineered_dfs[key].columns),
            'New Features': len(engineered_dfs[key].columns) - len(clean_dfs[key].columns)
        })

print(pd.DataFrame(comparison).set_index('Dataset').to_string())

## Summary

Feature preparation complete. All derived columns are ready for use in Phase 3 ML training.

### Features created:
- **Fleet**: `Vehicle_Age_Category`, `Fuel_Efficiency_Category`, `Readiness_Label`, +2 more
- **Battery**: `SOH_Category`, `Degradation_Severity`, `Battery_Age_Indicator`, +1 more
- **Charging**: `Fast_Charging_Flag`, `SOC_Delta_Computed`, `Cost_Per_kWh_Computed`, +2 more
- **Carbon**: `Carbon_Saving_Potential_kg`, `EV_Priority_Score`, `Net_Zero_Progress_Category`, +1 more
- **Weather**: `Temperature_Category`, `Climate_Severity_Index`, `Extreme_Weather_Flag`

**Phase 2 complete. Proceed to Phase 3: Machine Learning.**